# Analyze Lighthouse

Experimentation with analyzing Lighthouse scores.

In [3]:
import sys
import ast
from pathlib import Path

In [4]:
import numpy as np
import pandas as pd
import altair as alt

In [5]:
this_dir = Path("__file__").parent.absolute()

In [6]:
sys.path.append(this_dir.parent)

In [7]:
sys.path.append(str(this_dir.parent / "newshomepages"))

In [8]:
import altair_theme

In [9]:
alt.themes.register('palewire', altair_theme.theme)
alt.themes.enable('palewire')

ThemeRegistry.enable('palewire')

In [10]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [11]:
analysis_dir = this_dir.parent / "_analysis"

Read in the dataframe

In [12]:
df = pd.read_csv(
    extracts_dir / "lighthouse-sample.csv",
    usecols=[
        'handle',
        'file_name',
        'date',
        'performance',
        'accessibility',
        'seo',
        'best_practices',
    ],
    dtype={
        'handle': str,
        'file_name': str,
        'performance': float,
        'accessibility': float,
    },
    parse_dates=["date"]
)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10865 entries, 0 to 10864
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   handle          10865 non-null  object        
 1   file_name       10865 non-null  object        
 2   date            10865 non-null  datetime64[ns]
 3   performance     10863 non-null  float64       
 4   accessibility   10865 non-null  float64       
 5   best_practices  10747 non-null  float64       
 6   seo             10865 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(2)
memory usage: 594.3+ KB


Exclude any sites with less than 10 observations

In [14]:
observations_by_site = df.groupby("handle").size().rename("n").reset_index()

In [15]:
not_qualified = observations_by_site[observations_by_site.n < 10]

In [16]:
qualified_df = df[~df.handle.isin(not_qualified.handle)].copy()

Aggregate descriptive statistics for each metric.

In [17]:
agg_df = qualified_df.groupby("handle").agg({
    'performance': ['count', 'median', 'mean', 'min', 'max', 'std'],
    'accessibility': ['count', 'median', 'mean', 'min', 'max', 'std'],
    'seo': ['count', 'median', 'mean', 'min', 'max', 'std'],
    'best_practices': ['count', 'median', 'mean', 'min', 'max', 'std'],
})

In [18]:
agg_df

performance                                        accessibility  \
                   count median      mean   min   max       std         count   
handle                                                                          
100Reporters          13  0.160  0.163077  0.14  0.20  0.018879            13   
11AliveNews           13  0.220  0.219231  0.15  0.28  0.039468            13   
12NewsNow             13  0.170  0.213846  0.08  0.36  0.082718            13   
12khari               13  0.160  0.152308  0.04  0.31  0.066477            13   
13wmaznews            12  0.240  0.250000  0.14  0.38  0.085387            12   
...                  ...    ...       ...   ...   ...       ...           ...   
yorkdispatch          12  0.835  0.815833  0.70  0.86  0.051779            12   
zeitonline            12  0.495  0.486667  0.39  0.58  0.053824            12   
zerohedge             17  0.470  0.457059  0.36  0.54  0.055200            17   
zerohora              17  0.260  0.260000  0.24  0.29  0.012247            17   
zn_ua                 12  0.245  0.248333  0.21  0.31  0.032427            12   

                                     ...       seo                        \
             median      mean   min  ...      mean   min   max       std   
handle                               ...                                   
100Reporters   0.89  0.890000  0.89  ...  0.874615  0.87  0.88  0.005189   
11AliveNews    0.83  0.820769  0.81  ...  0.700000  0.70  0.70  0.000000   
12NewsNow      0.81  0.813077  0.81  ...  0.700000  0.70  0.70  0.000000   
12khari        0.78  0.780000  0.78  ...  0.850000  0.85  0.85  0.000000   
13wmaznews     0.83  0.821667  0.81  ...  0.776667  0.77  0.78  0.004924   
...             ...       ...   ...  ...       ...   ...   ...       ...   
yorkdispatch   0.88  0.883333  0.88  ...  0.930000  0.93  0.93  0.000000   
zeitonline     0.88  0.886667  0.86  ...  0.990000  0.99  0.99  0.000000   
zerohedge      0.95  0.950000  0.95  ...  0.898235  0.89  0.90  0.003930   
zerohora       0.84  0.840000  0.84  ...  0.910000  0.91  0.91  0.000000   
zn_ua          0.66  0.660000  0.66  ...  0.910000  0.91  0.91  0.000000   

             best_practices                                         
                      count median      mean   min   max       std  
handle                                                              
100Reporters             13   0.58  0.580000  0.58  0.58  0.000000  
11AliveNews              13   0.75  0.736923  0.58  0.75  0.047150  
12NewsNow                13   0.83  0.798462  0.58  0.83  0.079146  
12khari                  13   0.83  0.817692  0.75  0.83  0.030043  
13wmaznews               12   0.83  0.776667  0.67  0.83  0.071010  
...                     ...    ...       ...   ...   ...       ...  
yorkdispatch             12   0.92  0.920000  0.92  0.92  0.000000  
zeitonline               12   0.92  0.920000  0.92  0.92  0.000000  
zerohedge                17   0.92  0.914706  0.83  0.92  0.021828  
zerohora                 17   0.92  0.947059  0.83  1.00  0.058605  
zn_ua                    12   0.92  0.897500  0.83  0.92  0.040704  

[772 rows x 24 columns]

Flatten the dataframe

In [19]:
flat_df = agg_df.copy()
flat_df.columns = ['_'.join(col) for col in flat_df.columns]

In [20]:
flat_df.sort_values("performance_count")

,performance_count,performance_median,performance_mean,performance_min,performance_max,performance_std,accessibility_count,accessibility_median,accessibility_mean,accessibility_min,...,seo_mean,seo_min,seo_max,seo_std,best_practices_count,best_practices_median,best_practices_mean,best_practices_min,best_practices_max,best_practices_std
handle,,,,,,,,,,,,,,,,,,,,,
occrp,10,0.570,0.567000,0.53,0.61,0.023594,10,0.86,0.860000,0.86,...,0.840000,0.84,0.84,0.000000,10,0.92,0.920000,0.92,0.92,0.000000
tass_agency,10,1.000,0.908000,0.08,1.00,0.290930,10,0.71,0.714000,0.71,...,0.713000,0.69,0.92,0.072732,9,0.83,0.830000,0.83,0.83,0.000000
oronline,10,0.075,0.110000,0.05,0.21,0.063944,10,0.81,0.802000,0.79,...,0.750000,0.75,0.75,0.000000,10,0.75,0.809000,0.75,0.92,0.080478
EpochTimes,11,0.220,0.204545,0.00,0.26,0.071605,11,0.86,0.862727,0.84,...,0.772727,0.76,0.83,0.028316,11,0.75,0.742727,0.67,0.75,0.024121
wsbtv,11,0.440,0.369091,0.19,0.48,0.119955,11,0.72,0.736364,0.70,...,0.850000,0.85,0.85,0.000000,11,0.75,0.750000,0.75,0.75,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
foxnews,25,0.180,0.179200,0.14,0.23,0.016310,25,0.83,0.833200,0.83,...,0.820000,0.82,0.82,0.000000,25,0.83,0.830000,0.83,0.83,0.000000
ajc,28,0.090,0.099286,0.06,0.14,0.024934,28,0.68,0.680000,0.68,...,0.840000,0.84,0.84,0.000000,28,0.83,0.830000,0.83,0.83,0.000000
globeandmail,28,0.190,0.181786,0.10,0.28,0.051067,28,1.00,0.996429,0.98,...,0.975000,0.92,0.99,0.029250,28,0.92,0.920000,0.92,0.92,0.000000


Classify the scores

In [21]:
def color_code(val):
    """Return the classification of a metric according to Google's system.
    
    Source: https://developer.chrome.com/docs/lighthouse/performance/performance-scoring/
    """
    if val >= .9:
        return 'green'
    elif val >= .5:
        return 'orange'
    else:
        return 'red'

In [22]:
flat_df['performance_color'] = flat_df.performance_median.apply(color_code)

In [23]:
flat_df['accessibility_color'] = flat_df.accessibility_median.apply(color_code)

Rank the result

In [24]:
flat_df['performance_rank'] = flat_df.performance_median.rank(ascending=False, method="min")

In [25]:
flat_df.sort_values("performance_rank")[[
    'performance_median',
    'performance_rank'
]]

,performance_median,performance_rank
handle,,
tass_agency,1.000,1.0
insideclimate,1.000,1.0
techmeme,0.975,3.0
lobs,0.960,4.0
gigharbornow,0.960,4.0
...,...,...
qctimes,0.030,758.0
BostonHerald,0.020,769.0
portalimprensa,0.020,769.0


Total up the colors

In [26]:
flat_df.performance_color.value_counts()

red       601
orange    162
green       9
Name: performance_color, dtype: int64

In [27]:
flat_df.performance_color.value_counts(normalize=True)

red       0.778497
orange    0.209845
green     0.011658
Name: performance_color, dtype: float64

In [28]:
flat_df.accessibility_color.value_counts()

orange    531
green     237
red         4
Name: accessibility_color, dtype: int64

In [29]:
flat_df.accessibility_color.value_counts(normalize=True)

orange    0.687824
green     0.306995
red       0.005181
Name: accessibility_color, dtype: float64

In [30]:
flat_df.performance_median.describe()

count    772.000000
mean       0.348012
std        0.213881
min        0.010000
25%        0.190000
50%        0.280000
75%        0.470000
max        1.000000
Name: performance_median, dtype: float64

In [31]:
flat_df.accessibility_median.describe()

count    772.000000
mean       0.844585
std        0.099997
min        0.430000
25%        0.790000
50%        0.860000
75%        0.920000
max        1.000000
Name: accessibility_median, dtype: float64

In [32]:
chart_df = (
    qualified_df[qualified_df.handle == 'nytimes']
        .set_index(["handle", "file_name", "date"])
        .stack()
        .reset_index()
        .rename(columns={0: 'value', 'level_3': 'metric'})
)

In [33]:
chart_df['color'] = chart_df.value.apply(color_code)

In [34]:
chart_df.value = chart_df.value * 100

In [35]:
chart_df.metric = chart_df.metric.str.capitalize().str.replace("_" , " ").replace("Seo", "SEO")

In [36]:
chart_df.head()

,handle,file_name,date,metric,value,color
0,nytimes,nytimes-2022-08-11T01:19:38.394745-04:00.light...,2022-08-11,Performance,36.0,red
1,nytimes,nytimes-2022-08-11T01:19:38.394745-04:00.light...,2022-08-11,Accessibility,100.0,green
2,nytimes,nytimes-2022-08-11T01:19:38.394745-04:00.light...,2022-08-11,Best practices,83.0,orange
3,nytimes,nytimes-2022-08-11T01:19:38.394745-04:00.light...,2022-08-11,SEO,98.0,green
4,nytimes,nytimes-2022-08-11T12:55:19.137945-04:00.light...,2022-08-11,Performance,26.0,red


In [37]:
alt.Chart(chart_df).mark_tick(height=20, opacity=0.9).encode(
    x=alt.X('value:Q', axis=alt.Axis(title=None)),
    y=alt.Y('metric:O', title=None),
    color=alt.Color("color:N", legend=None, scale=alt.Scale(domain=["green", "orange", "red"], range=["green", "orange", "red"])),
    tooltip=["metric", "date", "value"]
).properties(
    title="Lighthouse scores over last 7 days",
    width=500,
    height=175
).configure_axisY(
    labelFontSize=14,
)

alt.Chart(...)

In [48]:
def _round(val):
    return np.floor(np.floor(val * 1000)/100)*10

In [49]:
_round(0.67)

60.0

In [50]:
flat_df['performance_decile'] = flat_df.performance_median.apply(_round)

In [52]:
flat_df.head()

,performance_count,performance_median,performance_mean,performance_min,performance_max,performance_std,accessibility_count,accessibility_median,accessibility_mean,accessibility_min,...,best_practices_count,best_practices_median,best_practices_mean,best_practices_min,best_practices_max,best_practices_std,performance_color,accessibility_color,performance_rank,performance_decile
handle,,,,,,,,,,,,,,,,,,,,,
100Reporters,13,0.16,0.163077,0.14,0.20,0.018879,13,0.89,0.890000,0.89,...,13,0.58,0.580000,0.58,0.58,0.000000,red,orange,633.0,10.0
11AliveNews,13,0.22,0.219231,0.15,0.28,0.039468,13,0.83,0.820769,0.81,...,13,0.75,0.736923,0.58,0.75,0.047150,red,orange,502.0,20.0
12NewsNow,13,0.17,0.213846,0.08,0.36,0.082718,13,0.81,0.813077,0.81,...,13,0.83,0.798462,0.58,0.83,0.079146,red,orange,609.0,10.0
12khari,13,0.16,0.152308,0.04,0.31,0.066477,13,0.78,0.780000,0.78,...,13,0.83,0.817692,0.75,0.83,0.030043,red,orange,633.0,10.0
13wmaznews,12,0.24,0.250000,0.14,0.38,0.085387,12,0.83,0.821667,0.81,...,12,0.83,0.776667,0.67,0.83,0.071010,red,orange,460.0,20.0


In [53]:
histogram_df = flat_df.performance_decile.value_counts().reset_index()

In [54]:
histogram_df['index'] = histogram_df['index'].astype(int)

In [58]:
list(range(0, 11))*10

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10]

In [55]:
alt.Chart(histogram_df).mark_bar(opacity=0.9).encode(
    alt.X("index:Q", axis=alt.Axis(format='.0f'), bin=True, title="Median score"),
    y=alt.Y('performance_decile:Q', title="Number of sites"),
).properties(
    title="Lighthouse performance scores",
    width=500
)

alt.Chart(...)